In [3]:
!pip -q install requests beautifulsoup4 pandas lxml

In [5]:
import requests, re
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

url = "https://www.prada.com/us/en/womens/bags/c/10062US"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

html = requests.get(url, headers=HEADERS, timeout=30).text
soup = BeautifulSoup(html, "lxml")

p_links = []
for a in soup.select("a[href]"):
    href = a["href"]
    if "/p/" in href:
        full = urljoin(url, href)
        last = urlparse(full).path.rstrip("/").split("/")[-1]
        p_links.append((href, last, a.get_text(" ", strip=True)[:80]))

print("Found /p/ links:", len(p_links))
for x in p_links[:10]:
    print(x)

Found /p/ links: 24
('https://www.prada.com/us/en/p/prada-passage-medium-leather-bag-with-re-nylon-flap/1BA495_2G52_F0002_V_OPO', '1BA495_2G52_F0002_V_OPO', 'FROM THE RUNWAY FROM THE RUNWAY Prada Passage medium leather bag with Re-Nylon f')
('https://www.prada.com/us/en/p/prada-passage-medium-leather-bag-with-re-nylon-flap/1BA495_2G52_F0201_V_OPO', '1BA495_2G52_F0201_V_OPO', 'Prada Passage medium leather bag with Re-Nylon flap $ 4,100 Coffee Black')
('https://www.prada.com/us/en/p/prada-route-large-leather-tote-bag/1BG644_2G6F_F0002_V_OOO', '1BG644_2G6F_F0002_V_OOO', 'Prada Route large leather tote bag $ 5,000 Black Cocoa Brown')
('https://www.prada.com/us/en/p/prada-route-medium-leather-tote-bag/1BG645_2G6F_F0002_V_OOO', '1BG645_2G6F_F0002_V_OOO', 'FROM THE RUNWAY FROM THE RUNWAY Prada Route medium leather tote bag $ 4,500 Blac')
('https://www.prada.com/us/en/p/prada-route-large-leather-top-handle-bag/1BB162_2HFQ_F0002_V_OOO', '1BB162_2HFQ_F0002_V_OOO', 'FROM THE RUNWAY FROM THE RUNWA

In [23]:
SELECTED_KEYS = [
    "1BC167_RV44_F0038_V_B1M", #79
    "1BG602_2HIK_F0036_V_OOO", #39
    "1N204X_R064_F0002", #28
    "1BD394_RDLN_F0040_V_NOO", #57
    "1BA426_2HIM_F0G3N_V_MOO", #97
    "1BC229_2CYS_F04XV_V_LVM", #42
    "1BZ039_RV44_F0002_V_DMM", #72
    "1BZ811_RV44_F0D57_V_OTM", #64
    "1BG644_2G6S_F0161_V_OOO", #12
    "1N204W_NZV_F0WHB", #26
]

In [29]:
import re, time, random, requests, pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

SEEDS = {
    "US": "https://www.prada.com/us/en/womens/bags/c/10062US",
    "FR": "https://www.prada.com/fr/en/womens/bags/c/10062EU",
    "IT": "https://www.prada.com/it/it/womens/bags/c/10062EU",
    "CN": "https://www.prada.cn/cn/en/womens/bags/c/10062CN",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

PRICE_RE = re.compile(r"([€$¥])\s*([0-9][0-9\.,]*)")

def polite_sleep(a=0.6, b=1.2):
    time.sleep(random.uniform(a, b))

def fetch(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def parse_price(text: str):
    if not text:
        return (None, None)
    t = " ".join(text.split())
    m = PRICE_RE.search(t)
    if not m:
        return (None, None)
    cur, num_raw = m.group(1), m.group(2)

    # normalize "4,100" -> 4100 ; "3.100" -> 3100 (EU thousands)
    if "," in num_raw and "." not in num_raw:
        num = float(num_raw.replace(",", ""))
    elif "." in num_raw and "," not in num_raw:
        parts = num_raw.split(".")
        if all(len(p) == 3 for p in parts[1:]):
            num = float("".join(parts))
        else:
            num = float(num_raw)
    else:
        num = float(num_raw.replace(",", ""))
    return cur, num

def product_key(u: str) -> str:
    """
    Robust key: last path segment of the /p/ product URL
    (works whether it has underscores or not).
    """
    path = urlparse(u).path.rstrip("/")
    return path.split("/")[-1].upper()

def extract_products_from_category(html: str, base_url: str):
    soup = BeautifulSoup(html, "lxml")
    out = {}

    for a in soup.select("a[href]"):
        href = a["href"]
        if "/p/" not in href:
            continue

        full = urljoin(base_url, href)
        key = product_key(full)

        # Text on Prada pages often includes name and price near the link
        text = a.get_text(" ", strip=True)
        cur, price = parse_price(text)

        # If link text is empty, fallback to parent text
        if not text and a.parent:
            text = a.parent.get_text(" ", strip=True)
            cur, price = parse_price(text)

        name = text.strip() if text else None
        if name and cur:
            # keep name before the currency symbol if possible
            name = name.split(cur)[0].strip() or name

        # Store first occurrence per key
        out.setdefault(key, {
            "key": key,
            "product_name": name,
            "currency_symbol": cur,
            "price": price,
            "product_url": full,
        })

    return list(out.values())

def get_n_from_seed(seed_url: str, n=10, max_pages=12):
    got, seen = [], set()
    for page in range(1, max_pages + 1):
        url = seed_url if page == 1 else f"{seed_url}/page/{page}"
        html = fetch(url)
        items = extract_products_from_category(html, seed_url)

        for it in items:
            if it["key"] not in seen:
                seen.add(it["key"])
                got.append(it)

        if len(got) >= n:
            break

        polite_sleep()

    return got[:n]

def build_pool(seed_url: str, target=300, max_pages=25):
    pool = {}
    for page in range(1, max_pages + 1):
        url = seed_url if page == 1 else f"{seed_url}/page/{page}"
        html = fetch(url)
        items = extract_products_from_category(html, seed_url)

        for it in items:
            pool.setdefault(it["key"], it)

        if len(pool) >= target:
            break

        polite_sleep()

    return pool

# 1) Anchor 10 from US
anchors = get_n_from_seed(SEEDS["US"], n=10)
print("US anchors found:", len(anchors))
if len(anchors) == 0:
    raise RuntimeError("Still found 0 anchors. Run Cell A and paste the first few /p/ links so we can target the right markup.")

#anchor_keys = [a["key"] for a in anchors]
anchor_keys = SELECTED_KEYS
print("Anchor keys:", anchor_keys)

# 2) Match across countries
rows = []
for country, seed in SEEDS.items():
    pool = build_pool(seed, target=400, max_pages=30)
    for rank, key in enumerate(anchor_keys, start=1):
        it = pool.get(key)
        rows.append({
            "brand": "Prada",
            "anchor_rank_us": rank,
            "key": key,
            "country": country,
            "product_name": it["product_name"] if it else None,
            "currency_symbol": it["currency_symbol"] if it else None,
            "price": it["price"] if it else None,
            "product_url": it["product_url"] if it else None,
            "status": "ok_from_category" if it else "not_found_in_country_listing",
        })

df = pd.DataFrame(rows)
out_csv = "prada_10bags_4countries.csv"
df.to_csv(out_csv, index=False)

print("Saved:", out_csv)
print("Shape:", df.shape)
df.head(12)

US anchors found: 10
Anchor keys: ['1BC167_RV44_F0038_V_B1M', '1BG602_2HIK_F0036_V_OOO', '1N204X_R064_F0002', '1BD394_RDLN_F0040_V_NOO', '1BA426_2HIM_F0G3N_V_MOO', '1BC229_2CYS_F04XV_V_LVM', '1BZ039_RV44_F0002_V_DMM', '1BZ811_RV44_F0D57_V_OTM', '1BG644_2G6S_F0161_V_OOO', '1N204W_NZV_F0WHB']
Saved: prada_10bags_4countries.csv
Shape: (40, 9)


,brand,anchor_rank_us,key,country,product_name,currency_symbol,price,product_url,status
0,Prada,1,1BC167_RV44_F0038_V_B1M,US,for SEA BEYOND for SEA BEYOND Re-Nylon shoulde...,$,2150.0,https://www.prada.com/us/en/p/re-nylon-shoulde...,ok_from_category
1,Prada,2,1BG602_2HIK_F0036_V_OOO,US,Large leather tote bag,$,4900.0,https://www.prada.com/us/en/p/large-leather-to...,ok_from_category
2,Prada,3,1N204X_R064_F0002,US,Prada Re-Edition 2005 Re-Nylon and Saffiano mi...,$,1390.0,https://www.prada.com/us/en/p/prada-re-edition...,ok_from_category
3,Prada,4,1BD394_RDLN_F0040_V_NOO,US,for SEA BEYOND for SEA BEYOND Prada Explore me...,$,2500.0,https://www.prada.com/us/en/p/prada-explore-me...,ok_from_category
4,Prada,5,1BA426_2HIM_F0G3N_V_MOO,US,Prada Bonnie medium printed leather handbag,$,3850.0,https://www.prada.com/us/en/p/prada-bonnie-med...,ok_from_category
5,Prada,6,1BC229_2CYS_F04XV_V_LVM,US,Prada Aimée medium leather shoulder bag,$,3300.0,https://www.prada.com/us/en/p/prada-aimee-medi...,ok_from_category
6,Prada,7,1BZ039_RV44_F0002_V_DMM,US,for SEA BEYOND for SEA BEYOND Re-Nylon backpack,$,2850.0,https://www.prada.com/us/en/p/re-nylon-backpac...,ok_from_category
7,Prada,8,1BZ811_RV44_F0D57_V_OTM,US,Medium Re-Nylon Backpack,$,2500.0,https://www.prada.com/us/en/p/medium-re-nylon-...,ok_from_category
8,Prada,9,1BG644_2G6S_F0161_V_OOO,US,Prada Route large canvas and leather tote bag,$,3200.0,https://www.prada.com/us/en/p/prada-route-larg...,ok_from_category
9,Prada,10,1N204W_NZV_F0WHB,US,Prada Re-Edition 2005 Saffiano leather mini-bag,$,2100.0,https://www.prada.com/us/en/p/prada-re-edition...,ok_from_category


In [17]:
# Dataset with all bags (will randomly select 10 to use from this set)
us_pool = build_pool(SEEDS["US"], target=80, max_pages=10)

us_df = pd.DataFrame(us_pool.values())
us_df[["key", "product_name", "price"]].head(157)
out_csv = "prada_157bags.csv"
us_df.to_csv(out_csv, index=False)